# Candidate DFT comparison

Source: archived main notebook, cells 105–111. The original 13-row DFT CSV is absent. The retained `*_DFT_pred.csv` supplies input and DFT columns; its old prediction columns are ignored. Three rows were previously reported, leaving ten new candidates. Reported status requires the verified ledger.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import numpy as np
import pandas as pd
from _modeling_common import (PHYSORG_FEATURE_COUNT, TARGET_COLUMNS,
    broad_training_ids, evaluate_targets, fit_delta_g_feature,
    load_modeling_data, metrics, new_catboost)

data_Ga, X_Ga, data_G, X_G = load_modeling_data(DATA_DIR, DESCRIPTOR_MAP)
Y_DD = data_Ga["Diene_Distort"].to_numpy()
Y_ED = data_Ga["Ene_Distort"].to_numpy()
Y_IN = data_Ga["Interaction"].to_numpy()
Y_Ga = data_Ga["deltaGa(Solvent)"].to_numpy()
Y_G = data_Ga["deltaG"].to_numpy()
Y_G_data_G = data_G["deltaG"].to_numpy()
print(f"Aligned active-energy rows: {len(data_Ga):,}; broad rows: {len(data_G):,}")

In [ ]:
from Code import model_train
from sklearn.model_selection import KFold

CANDIDATE_FILE = (REPO_ROOT / "Data" / "Predict_DI" /
                  "6_DI_selected_Purchasable_DFT_pred.csv")
candidate_raw = pd.read_csv(CANDIDATE_FILE).reset_index(drop=True)
if len(candidate_raw) != 13:
    raise ValueError(f"Expected 13 candidate rows, found {len(candidate_raw)}")
candidate_features, retained = model_train.read_df(candidate_raw, str(DESCRIPTOR_MAP))
candidates = candidate_raw.iloc[retained].reset_index(drop=True)
if len(candidates) != 13 or len(candidate_features) != 13:
    raise ValueError("The selected descriptor map does not cover all 13 candidates")
if "Structure_id" not in candidates and "Structure_Id" in candidates:
    candidates = candidates.rename(columns={"Structure_Id": "Structure_id"})

In [ ]:
candidate_rows = []
prediction_table = candidates[["Diene_Index", "Ene_Index", "Structure_id"]].copy()
targets = ["Diene_Distort", "Ene_Distort", "deltaGa(Solvent)"]
for variant in ["PhysOrg", "PhysOrg + SVO", "PhysOrg + SVO + predicted reaction energy"]:
    if variant == "PhysOrg":
        train_features = X_Ga[:, :PHYSORG_FEATURE_COUNT]
        test_features = candidate_features[:, :PHYSORG_FEATURE_COUNT]
    elif variant == "PhysOrg + SVO":
        train_features, test_features = X_Ga, candidate_features
    else:
        crossfit_g = np.full(len(data_Ga), np.nan)
        for train_ids, test_ids in KFold(5, shuffle=True, random_state=0).split(X_Ga):
            g_ids = broad_training_ids(data_G, data_Ga, test_ids)
            g_model = new_catboost()
            g_model.fit(X_G[g_ids], Y_G_data_G[g_ids])
            crossfit_g[test_ids] = g_model.predict(X_Ga[test_ids])
        g_model = new_catboost()
        g_model.fit(X_G, Y_G_data_G)
        train_features = np.column_stack([X_Ga, crossfit_g])
        test_features = np.column_stack([candidate_features,
                                         g_model.predict(candidate_features)])
    for target in targets:
        model = new_catboost()
        model.fit(train_features, data_Ga[target].to_numpy())
        predicted = model.predict(test_features)
        prediction_table[f"{variant} | {target}"] = predicted
        candidate_rows.append({"variant": variant, "target": target,
                               "n": len(candidates),
                               **metrics(candidates[target].to_numpy(), predicted)})
comparison = pd.DataFrame(candidate_rows)
display(comparison)
display(prediction_table)

No 1.89 kcal mol⁻¹ standard-state correction is applied to this model-versus-DFT comparison.
